# 01 - NEPSE Data Cleaning

Clean the NEPSE index file and produce `data_nepse_cleaned.csv`.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_FILE = Path("nepse_index_2015-01-01_to_2026-06-03.csv")
OUT = Path("data_nepse_cleaned.csv")

df = pd.read_csv(DATA_FILE, parse_dates=["date_ad"])
original_rows = len(df)
df = df.sort_values("date_ad")
df.head()


,date_ad,index_value,absolute_change,percentage_change
0,2015-01-01,903.68,1.35,0.15
1,2015-01-04,917.50,13.82,1.53
2,2015-01-05,919.77,2.27,0.25
3,2015-01-06,917.41,-2.36,-0.26
4,2015-01-07,924.54,7.13,0.78


In [2]:
audit = {
    "original_rows": original_rows,
    "date_min": df["date_ad"].min().date(),
    "date_max": df["date_ad"].max().date(),
    "duplicate_dates": int(df["date_ad"].duplicated().sum()),
    "missing_index_values": int(df["index_value"].isna().sum()),
}
audit


{'original_rows': 2596,
 'date_min': datetime.date(2015, 1, 1),
 'date_max': datetime.date(2026, 6, 3),
 'duplicate_dates': 0,
 'missing_index_values': 0}

In [3]:
df = df.drop_duplicates(subset=["date_ad"], keep="last")
df = df.dropna(subset=["date_ad", "index_value"])

for col in ["index_value", "absolute_change", "percentage_change"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["index_value"])
df["calendar_gap_days"] = df["date_ad"].diff().dt.days
df["is_long_gap"] = df["calendar_gap_days"] > 4

clean_audit = {
    "cleaned_rows": len(df),
    "removed_rows": original_rows - len(df),
    "long_calendar_gaps_over_4_days": int(df["is_long_gap"].sum()),
}
clean_audit


{'cleaned_rows': 2596, 'removed_rows': 0, 'long_calendar_gaps_over_4_days': 40}

In [4]:
df.to_csv(OUT, index=False)
print(f"Saved {len(df):,} cleaned rows to {OUT.resolve()}")
df.tail()


Saved 2,596 cleaned rows to /Users/anamgiri/Desktop/Nepal stock market/data_nepse_cleaned.csv


,date_ad,index_value,absolute_change,percentage_change,calendar_gap_days,is_long_gap
2591,2026-05-26,2777.10,-9.24,-0.33,1.0,False
2592,2026-05-27,2782.10,4.99,0.17,1.0,False
2593,2026-06-01,2755.37,-26.72,-0.96,5.0,True
2594,2026-06-02,2777.58,22.21,0.80,1.0,False
2595,2026-06-03,2780.25,2.66,0.09,1.0,False
